### Homework 5: Pandas Practice

For this homework, you'll be using Pandas to analyze a database of fatal police shootings in the US.

In [1]:
import pandas as pd

In [2]:
data = pd.read_csv("fatal-police-shootings-data.csv",
                   parse_dates=["date"])

In [3]:
data.dtypes

id                                     int64
date                          datetime64[us]
threat_type                              str
flee_status                              str
armed_with                               str
city                                     str
county                                   str
state                                    str
latitude                             float64
longitude                            float64
location_precision                       str
name                                     str
age                                  float64
gender                                   str
race                                     str
race_source                              str
was_mental_illness_related              bool
body_camera                             bool
agency_ids                               str
dtype: object

In [4]:
data.head()

,id,date,threat_type,flee_status,armed_with,city,county,state,latitude,longitude,location_precision,name,age,gender,race,race_source,was_mental_illness_related,body_camera,agency_ids
0,3,2015-01-02,point,not,gun,Shelton,Mason,WA,47.246826,-123.121592,not_available,Tim Elliot,53.0,male,A,not_available,True,False,73
1,4,2015-01-02,point,not,gun,Aloha,Washington,OR,45.487421,-122.891696,not_available,Lewis Lee Lembke,47.0,male,W,not_available,False,False,70
2,5,2015-01-03,move,not,unarmed,Wichita,Sedgwick,KS,37.694766,-97.280554,not_available,John Paul Quintero,23.0,male,H,not_available,False,False,238
3,8,2015-01-04,point,not,replica,San Francisco,San Francisco,CA,37.762910,-122.422001,not_available,Matthew Hoffman,32.0,male,W,not_available,True,False,196
4,9,2015-01-04,point,not,other,Evans,Weld,CO,40.383937,-104.692261,not_available,Michael Rodriguez,39.0,male,H,not_available,False,False,473


Spend some time examining the columns in the data to get a little familiar with the information you're working with...

In [5]:
len(data)

10430

In [6]:
# What is the time range of the available data? Use a line of pandas code to find out. 
data["date"].describe()

count                         10430
mean     2020-03-06 05:37:25.599233
min             2015-01-02 00:00:00
25%             2017-08-25 00:00:00
50%             2020-04-12 00:00:00
75%             2022-10-02 00:00:00
max             2024-12-31 00:00:00
Name: date, dtype: object

In [7]:
# Say you want to analyze the racial disparities in this dataset, broken down by race and by the larger category of non-white people. 
# Write a line of code that takes a look at the universe of values in the race column.

data["race"].unique()


<StringArray>
['A', 'W', 'H', 'B', 'O', nan, 'N', 'W;B;N', 'N;H', 'W;H', 'B;H', 'W;B',
 'W;A']
Length: 13, dtype: str

In [8]:
# Make the letters in the race column all lower-case
data["race"] = data["race"].str.replace("A","a").str.replace("W","w").str.replace("H","h").str.replace(
    "B","b").str.replace("O","o").str.replace("N","n").str.replace("W;B;N","w;b;n").str.replace(
    "N;H","n:h").str.replace("W;H","w;h").str.replace("B;H","b;h").str.replace("W;B","w;b").str.replace("W;A","w;a")
data["race"].unique()

<StringArray>
['a', 'w', 'h', 'b', 'o', nan, 'n', 'w;b;n', 'n;h', 'w;h', 'b;h', 'w;b',
 'w;a']
Length: 13, dtype: str

In [9]:
# Create a new column called "poc" and use a lambda function to set it equal to True or False based on the information in the race column.
data["poc"] = data["race"].apply(lambda x: x != 'w')

# How many rows are POC and how many are White?"
poc_rows = (data["poc"] == True).sum()
white_rows = (data["poc"] == False).sum()

print(f'There are {poc_rows} "poc" rows and {white_rows} "white" rows.')

There are 5771 "poc" rows and 4659 "white" rows.


In [10]:
# I want to know which county had the most police killings but some counties in the U.S. have the same name. 
# Create a column called country_state and populate it with the county and the state to create unique pairs. 
data["county_state"] = data["county"] + ", " + data["state"]
# Now answer the question: which county had the most killings?
county_killings_count = data["county_state"].value_counts(ascending = False)
top_county_killings = county_killings_count.index[0]
top_county_killings_number = county_killings_count.values[0]

print(f"There were the most killings in {top_county_killings} with {top_county_killings_number} killings.")

There were the most killings in Los Angeles, CA with 222 killings.


In [24]:
# Create a new column called "unarmed" and use a lambda function to set it to True/False based on the relevant info.
data["unarmed"] = data["armed_with"].apply(lambda x: "unarmed" in str(x))
# Create a subset of the data for unarmed killings and store it in a new variable called unarmed_killings. 
unarmed_killings = data[data["armed_with"] == "unarmed"]
# Which county had the most of these?
unarmed_county_killings_count = unarmed_killings["county_state"].value_counts(ascending = False)
top_unarmed_k_county = unarmed_county_killings_count.index[0]
top_unarmed_k_county_count = unarmed_county_killings_count.values[0]

print(f"There were the most unarmed killings in {top_unarmed_k_county} with {top_unarmed_k_county_count} killings.")

There were the most unarmed killings in Los Angeles, CA with 19 killings.


In [12]:
# We could have done that more smoothly using a groupby. So let's do that. 
# Group the original dataset by your new columns unarmed killings and county_state. 
data_grouped = data.groupby(
    ["county_state"]
)["unarmed"].value_counts( 
).reset_index().sort_values(by="count", ascending = False)

# Use loc[] to isolate the rows where unarmed==True
data_unarmed = data_grouped.loc[data_grouped['unarmed'] == True]
data_unarmed

# Sort from highest #killings to lowest

data_unarmed_sorted = data_unarmed.sort_values("unarmed", ascending = True)
data_unarmed_sorted

,county_state,unarmed,count
843,"Los Angeles, CA",True,19
728,"Kenai Peninsula, AK",True,1
425,"El Paso, CO",True,1
934,"Middlesex, MA",True,1
433,"Ellis, KS",True,1
...,...,...,...
1188,"Randall, TX",True,1
251,"Clackamas, OR",True,1
1224,"Rockingham, NC",True,1
1350,"St. Martin, LA",True,1


I want to examine the data by race and gender to understand whether Black women are disproportionately killed by the police compared to white women and by how much. I'm not going to tell you what to do beyond that — this is where you need to think creatively as a data reporter! What information do you need to look up to figure out this answer? Go step by step and show your working using Python comments.

In [25]:
# I'm going to create a subset of data with only the "female" victims.
female_victims = data[data["gender"] == "female"]
# female_victims

# I'm going to see what races appear in this subset.
female_victims["race"].unique()
    # I now know that the only race indicator that includes 'b' is 'b' alone. I'll check for all of them just in case tho...

# Now, I'm going to group them by race. 
black_victim_codes = ['b', 'w;b;n', 'b;h', 'w;b']
female_black = female_victims.loc[female_victims['race'].isin(black_victim_codes)]
female_white = female_victims.loc[female_victims['poc'] == False]

# Now I'm going to count them.
f_black_count = len(female_black)
f_white_count = len(female_white)

#f string time!

print(f"There are {f_black_count} Black women who have been killed by police, while {f_white_count} white women have been killed by police.")

There are 88 Black women who have been killed by police, while 254 white women have been killed by police.


462

Your editor wants you to examine mental illness in police killings in the US by state. Seeing which state had the most killings of mentally ill people is not enough; you'll need to scale by population. To do that, you will use the Python census library to access Census data. You'll need an API key to use it.

Sign up for a Census API key at this website: https://api.census.gov/data/key_signup.html 

Then check your email to get the API key. Make sure to click the activation link in the email.

In [14]:
# install the python census library.
%pip install census
%pip install geonamescache

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


### TAKE A BREAK!

It can take several minutes for your API key to be activated. Go have a stretch, get some water, etc...

In [15]:
api_key = "ff71da105f49d6ec2cdd907b02771cf46b55165f" 

In [16]:
import os
from census import Census
from us import states
import geonamescache

In [17]:
gc = geonamescache.GeonamesCache()
us_states = gc.get_us_states()

# The geonamescache library stores the fips code for each state. 
# Adjust the loop I have below to populate the empty dictionary below, 
# where the keys are state codes (i.e. AK) and the values are fips codes (as strings)

state_to_fips = {}


for state_code, state_info in us_states.items():
    #print(state_info)
    state_to_fips[state_code] = state_info['fips']
state_to_fips


{'AK': '02',
 'AL': '01',
 'AR': '05',
 'AZ': '04',
 'CA': '06',
 'CO': '08',
 'CT': '09',
 'DC': '11',
 'DE': '10',
 'FL': '12',
 'GA': '13',
 'HI': '15',
 'IA': '19',
 'ID': '16',
 'IL': '17',
 'IN': '18',
 'KS': '20',
 'KY': '21',
 'LA': '22',
 'MA': '25',
 'MD': '24',
 'ME': '23',
 'MI': '26',
 'MN': '27',
 'MO': '29',
 'MS': '28',
 'MT': '30',
 'NC': '37',
 'ND': '38',
 'NE': '31',
 'NH': '33',
 'NJ': '34',
 'NM': '35',
 'NV': '32',
 'NY': '36',
 'OH': '39',
 'OK': '40',
 'OR': '41',
 'PA': '42',
 'RI': '44',
 'SC': '45',
 'SD': '46',
 'TN': '47',
 'TX': '48',
 'UT': '49',
 'VA': '51',
 'VT': '50',
 'WA': '53',
 'WI': '55',
 'WV': '54',
 'WY': '56'}

In [18]:
# Ping the Census API with your key...
c = Census(api_key, year=2023)

# We're using American Community Survey Version 5 (acs5 in the census library)
# Population is represented by data code "B01003_001E"
# So to pull the population for New York state, which is FIPS code 36, here's what you do:

ny_population_query = c.acs5.state(("NAME", "B01003_001E"), "36" )
print(ny_population_query)

# Examine the results of the query. Looks like the Census API hands back a list with a dictionary in it, 
# and you have to pull the population info out of that...

[{'NAME': 'New York', 'B01003_001E': 19872319.0, 'state': '36'}]


In [19]:
print(f"The population of New York is: {ny_population_query[0]['B01003_001E']}")

The population of New York is: 19872319.0


In [20]:
# OK now use the state_to_fips dictionary you made in conjunction with the census API to create one more dictionary
# where keys are the state code and values are the population, ie. {"AK":3423908,"AL": .... }

# This might take a while to run! You're pinging an API 50 times.  

# Your loop here!

state_populations = {}

for state_code, fips in state_to_fips.items():
    fips_population = c.acs5.state('B01003_001E', fips)
    state_populations[state_code] = int(fips_population[0]['B01003_001E'])
    
state_populations


{'AK': 733971,
 'AL': 5054253,
 'AR': 3032651,
 'AZ': 7268175,
 'CA': 39242785,
 'CO': 5810774,
 'CT': 3598348,
 'DC': 672079,
 'DE': 1005872,
 'FL': 21928881,
 'GA': 10822590,
 'HI': 1445635,
 'IA': 3195937,
 'ID': 1893296,
 'IL': 12692653,
 'IN': 6811752,
 'KS': 2937569,
 'KY': 4510725,
 'LA': 4621025,
 'MA': 6992395,
 'MD': 6170738,
 'ME': 1377400,
 'MI': 10051595,
 'MN': 5713716,
 'MO': 6168181,
 'MS': 2951438,
 'MT': 1105072,
 'NC': 10584340,
 'ND': 779361,
 'NE': 1965926,
 'NH': 1387834,
 'NJ': 9267014,
 'NM': 2114768,
 'NV': 3141000,
 'NY': 19872319,
 'OH': 11780046,
 'OK': 3995260,
 'OR': 4238714,
 'PA': 12986518,
 'RI': 1095371,
 'SC': 5212774,
 'SD': 899194,
 'TN': 6986082,
 'TX': 29640343,
 'UT': 3331187,
 'VA': 8657499,
 'VT': 645254,
 'WA': 7740984,
 'WI': 5892023,
 'WV': 1784462,
 'WY': 579761}

I want you to learn to use the Census API because, chances are, it will be useful data to merge/explore as you work on your projects. 

OK. Now back to our data on police shootings. Use pandas and the census library to figure out which state has the highest rate of police killings of disabled people. There isn't one way to get the answer! Think creatively. 

In [21]:
# Your code here

In [34]:
# Reducing my police data to just include killings of disabled people. 
disabled_killings = data[data["was_mental_illness_related"] == True]
disabled_killings

# Now I have a subset of these killings. Now I have to merge to figure out which state has the highest rate of police killings of disabled 
# people. 

# I need to extract the number of police killings per state from this data.

state_counts = disabled_killings['state'].value_counts()
state_counts

# Now, for every number of killings per state, I have to figure out the rate of killings. 

# I'm not sure how to deal with a series in Pandas, I can't figure out this final question. :(
    


state
CA    274
TX    168
FL    166
OH     72
WA     70
AZ     70
NC     64
GA     63
NY     63
TN     61
PA     52
OK     51
MI     49
IN     49
CO     47
OR     44
VA     43
AL     42
MO     42
WI     38
NV     37
LA     36
IL     35
NM     35
MD     34
SC     30
MN     29
UT     29
NJ     22
KY     20
AR     20
KS     20
MA     19
MS     17
WV     16
NH     14
SD     12
IA     12
ID     12
NE     11
AK     10
MT      9
ME      8
HI      8
WY      6
CT      6
VT      6
ND      6
DC      5
DE      4
RI      1
Name: count, dtype: int64